#Loading Prepared Data

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
notebook_path = '/content/drive/My Drive/Python Projects/Walmart Sales Forecast'
save_path = os.path.join(notebook_path, 'data')

In [ ]:
train_file = os.path.join(save_path, "train_merged.csv")

In [ ]:
train = pd.read_csv(train_file)

Check

In [ ]:
train.shape

(421570, 17)

In [ ]:
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,0,151315
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,1,0,151315
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0,0,151315
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0,0,151315
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0,0,151315


###Date/Time Features

In [ ]:
df = train.copy()
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter

In [ ]:
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
df['IsQuarterStart'] = df['Date'].dt.is_quarter_start.astype(int)
df['IsQuarterEnd'] = df['Date'].dt.is_quarter_end.astype(int)

In [ ]:
# Cyclic encoding
df['Month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month']/12)
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek']/7)
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek']/7)

###Holiday/Event Features

In [ ]:
# Holiday Dates
super_bowl = ['2010-02-12','2011-02-11','2012-02-10']
labor_day = ['2010-09-10','2011-09-09','2012-09-07']
thanksgiving = ['2010-11-26','2011-11-25','2012-11-23']
christmas = ['2010-12-31','2011-12-30','2012-12-28']

In [ ]:
df['SuperBowl'] = df['Date'].isin(pd.to_datetime(super_bowl)).astype(int)
df['LaborDay'] = df['Date'].isin(pd.to_datetime(labor_day)).astype(int)
df['Thanksgiving'] = df['Date'].isin(pd.to_datetime(thanksgiving)).astype(int)
df['Christmas'] = df['Date'].isin(pd.to_datetime(christmas)).astype(int)

In [ ]:
# Holiday proximity
events = pd.to_datetime(super_bowl + labor_day + thanksgiving + christmas)
df['DaysToHoliday'] = df['Date'].apply(lambda x: min((events - x).days, key=abs))

In [ ]:
# Holiday season (Nov-Dec)
df['IsHolidaySeason'] = df['Month'].isin([11, 12]).astype(int)

###Store-level Features

In [ ]:
df['SalesPerSize'] = df['Weekly_Sales'] / df['Size']
df['LogSize'] = np.log1p(df['Size'])

In [ ]:
# Rolling stats at store level
df = df.sort_values(['Store', 'Date'])
df['Store_Mean_Sales_4w'] = df.groupby('Store')['Weekly_Sales'].transform(lambda x: x.rolling(4,1).mean())
df['Store_Mean_Sales_12w'] = df.groupby('Store')['Weekly_Sales'].transform(lambda x: x.rolling(12,1).mean())
df['Store_Std_Sales_4w'] = df.groupby('Store')['Weekly_Sales'].transform(lambda x: x.rolling(4,1).std())

###Department-level Features

In [ ]:
df['Dept_Mean_Sales_4w'] = df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.rolling(4,1).mean())
df['Dept_Mean_Sales_12w'] = df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.rolling(12,1).mean())

In [ ]:
# Department holiday interaction
df['Dept_Holiday_Impact'] = df['Dept_Mean_Sales_4w'] * df['IsHoliday_x']

###Lag & Rolling Window Features

In [ ]:
for lag in [1,2,4]:
    df[f'Sales_lag{lag}'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(lag)

In [ ]:
for win in [4,8,12]:
    df[f'Sales_rollmean_{win}'] = df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.rolling(win,1).mean())
    df[f'Sales_rollstd_{win}'] = df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.rolling(win,1).std())

In [ ]:
# EWM
df['Sales_EWM_4w'] = df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.ewm(span=4).mean())

###Price & Promotion Features

In [ ]:
for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
    safe = df[col].clip(lower=0).fillna(0)
    df[f'{col}_log'] = np.log1p(safe)
    df[f'{col}_lag1'] = df.groupby('Store')[col].shift(1)

In [ ]:
df['PromoIntensity'] = df[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)
df['FuelPrice_Change'] = df['Fuel_Price'].pct_change()

###External Economic Features

In [ ]:
df['CPI_lag4'] = df['CPI'].shift(4)
df['Unemployment_lag4'] = df['Unemployment'].shift(4)

In [ ]:
df['CPI_pct_change'] = df['CPI'].pct_change(4)
df['Unemp_pct_change'] = df['Unemployment'].pct_change(4)

In [ ]:
df['EconStressIndex'] = df['CPI'] * df['Unemployment']

###Interaction Features

In [ ]:
df['StoreDept'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df['Holiday_Dept'] = df['Dept'].astype(str) + '_' + df['IsHoliday_x'].astype(str)
df['Holiday_Month'] = df['Month'].astype(str) + '_' + df['IsHoliday_x'].astype(str)

###Target Transformation

In [ ]:
df = df[df['Weekly_Sales'] >= 0].copy()
df['LogWeeklySales'] = np.log1p(df['Weekly_Sales'])

###Feature Scaling/Encoding

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scale_cols = ['Temperature','Fuel_Price','CPI','Unemployment','Size','PromoIntensity']
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

#Feature Selection & Regularization

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFECV
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

###Drop Collinear Features (VIF)

In [ ]:
def calculate_vif(df, features):
    X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    vif_data = pd.DataFrame({'feature': features, 'VIF': np.nan})

    for i in range(len(features)):
        try:
            vif_val = variance_inflation_factor(X.values, i)
            # If r²=1, this gives inf → keep as inf
            vif_data.loc[i, 'VIF'] = vif_val
        except Exception:
            vif_data.loc[i, 'VIF'] = np.inf

    return vif_data

In [ ]:
# numeric columns only (drop target + categorical IDs)
features = [col for col in df.select_dtypes(include=[np.number]).columns
            if col not in ['Weekly_Sales','LogWeeklySales']]

In [ ]:
vif_df = calculate_vif(df, features)
high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist()
df_vif_reduced = df.drop(columns=high_vif)   # Drop highly collinear
print("Dropped (high VIF):", high_vif)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero

Dropped (high VIF): ['IsHoliday_x', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday_y', 'Size', 'Year', 'Month', 'WeekOfYear', 'Quarter', 'Month_sin', 'SuperBowl', 'LaborDay', 'Thanksgiving', 'Christmas', 'LogSize', 'Dept_Mean_Sales_4w', 'Dept_Mean_Sales_12w', 'Sales_lag1', 'Sales_lag2', 'Sales_lag4', 'Sales_rollmean_4', 'Sales_rollmean_8', 'Sales_rollstd_8', 'Sales_rollmean_12', 'Sales_rollstd_12', 'Sales_EWM_4w', 'MarkDown1_log', 'MarkDown1_lag1', 'MarkDown2_lag1', 'MarkDown3_lag1', 'MarkDown4_log', 'MarkDown4_lag1', 'MarkDown5_log', 'MarkDown5_lag1', 'PromoIntensity', 'CPI_lag4', 'Unemployment_lag4', 'CPI_pct_change', 'Unemp_pct_change', 'EconStressIndex']


###Feature Importance (Tree-based Model)

In [ ]:
X = df_vif_reduced.drop(columns=['Weekly_Sales','LogWeeklySales'], errors='ignore')
X = X.select_dtypes(include=['int64','float64'])

y = df_vif_reduced['Weekly_Sales']

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 15 important features:\n", importances.head(15))

Top 15 important features:
 SalesPerSize            0.660040
Store_Std_Sales_4w      0.232715
Store_Mean_Sales_4w     0.040152
Type                    0.036723
Sales_rollstd_4         0.009954
Store                   0.009405
Dept                    0.005190
Store_Mean_Sales_12w    0.004739
Dept_Holiday_Impact     0.000431
Temperature             0.000293
Fuel_Price              0.000119
Month_cos               0.000057
MarkDown3_log           0.000056
DaysToHoliday           0.000052
MarkDown2_log           0.000038
dtype: float64


In [ ]:
# Drop low-importance features
low_imp_features = importances[importances < 0.001].index.tolist()
df_imp = df_vif_reduced.drop(columns=low_imp_features)
print("Dropped (low importance):", low_imp_features)

Dropped (low importance): ['Dept_Holiday_Impact', 'Temperature', 'Fuel_Price', 'Month_cos', 'MarkDown3_log', 'DaysToHoliday', 'MarkDown2_log', 'IsHolidaySeason', 'FuelPrice_Change', 'IsMonthEnd', 'IsQuarterStart', 'IsQuarterEnd', 'IsMonthStart', 'DayOfWeek_cos', 'DayOfWeek_sin']


In [ ]:
df_imp.head()

,Store,Dept,Date,Weekly_Sales,Type,DayOfWeek,SalesPerSize,Store_Mean_Sales_4w,Store_Mean_Sales_12w,Store_Std_Sales_4w,Sales_rollstd_4,StoreDept,Holiday_Dept,Holiday_Month,LogWeeklySales
0,1,1,2010-02-05,24924.50,0,4,0.164719,24924.5000,24924.5000,NaN,NaN,1_1,1_0,2_0,10.123647
143,1,2,2010-02-05,50605.27,0,4,0.334437,37764.8850,37764.8850,18159.046613,NaN,1_2,2_0,2_0,10.831831
286,1,3,2010-02-05,13740.12,0,4,0.090805,29756.6300,29756.6300,18901.638325,NaN,1_3,3_0,2_0,9.528148
429,1,4,2010-02-05,39954.04,0,4,0.264045,32305.9825,32305.9825,16253.555927,NaN,1_4,4_0,2_0,10.595510
572,1,5,2010-02-05,32229.38,0,4,0.212995,34132.2025,32290.6620,15542.559923,NaN,1_5,5_0,2_0,10.380665


###Recursive Feature Elimination (with CV)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)

In [ ]:
rfecv = RFECV(estimator=xgb, step=1, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
rfecv.fit(X, y)

RFECV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
      estimator=XGBRegressor(base_score=None, booster=None, callbacks=None,
                             colsample_bylevel=None, colsample_bynode=None,
                             colsample_bytree=None, device=None,
                             early_stopping_rounds=None,
                             enable_categorical=False, eval_metric=None,
                             feature_types=None, feature_weights=None,
                             gamma=None, grow_p...ance_type=None,
                             interaction_constraints=None, learning_rate=0.05,
                             max_bin=None, max_cat_threshold=None,
                             max_cat_to_onehot=None, max_delta_step=None,
                             max_depth=6, max_leaves=None,
                             min_child_weight=None, missing=nan,
                             monotone_constraints=None, multi_strategy=None,
                             n_estimators=200, n_jobs=None,
                             num_parallel_tree=None, ...),
      n_jobs=-1, scoring='neg_mean_squared_error')

In [ ]:
selected_features = X.columns[rfecv.support_].tolist()
print("Selected Features after RFE:", selected_features)

Selected Features after RFE: ['Type', 'SalesPerSize', 'Store_Mean_Sales_4w', 'Store_Std_Sales_4w']


###Best Features

In [ ]:
final_df = df_imp[selected_features + ['Weekly_Sales']]

In [ ]:
final_df.head()

,Type,SalesPerSize,Store_Mean_Sales_4w,Store_Std_Sales_4w,Weekly_Sales
0,0,0.164719,24924.5000,NaN,24924.50
143,0,0.334437,37764.8850,18159.046613,50605.27
286,0,0.090805,29756.6300,18901.638325,13740.12
429,0,0.264045,32305.9825,16253.555927,39954.04
572,0,0.212995,34132.2025,15542.559923,32229.38


##Saving the Data

In [ ]:
save_path = '/content/drive/My Drive/Python Projects/Walmart Sales Forecast/data'
print("Saving in:", save_path)

Saving in: /content/drive/My Drive/Python Projects/Walmart Sales Forecast/data


In [ ]:
os.makedirs(save_path, exist_ok=True)

df1_path  = os.path.join(save_path, "df_imp.csv")
df2_path  = os.path.join(save_path, "final_df.csv")

df_imp.to_csv(df1_path, index=False)
final_df.to_csv(df2_path, index=False)

print("Files saved at:", df1_path , "and", df2_path )

Files saved at: /content/drive/My Drive/Python Projects/Walmart Sales Forecast/data/df_imp.csv and /content/drive/My Drive/Python Projects/Walmart Sales Forecast/data/final_df.csv
